In [48]:
from pathlib import Path
import numpy as np


def read_sounding(file):

    with open(file) as f:
        lines = f.readlines()

    header = lines[1].split()

    df = pd.read_csv(
        file,
        sep=r"\s*\|\s*",
        engine="python",
        skiprows=3,
        header=None,
        names=header
    )

    return df
df = read_sounding("/users/jdelbeke/soundings/20240101_06610_data.txt")

print(df["termin"].unique())

from scipy.signal import savgol_filter

def detect_inversion(
    df,
    max_height=1000,
    edge_buffer_m=50,
    grid_res=20,
    threshold=0.01,
    min_depth=100
):

    profile = df.loc[df["level"] <= max_height, ["level","745"]]
    profile = profile.dropna(subset=["level", "745"])

    profile = (
        profile.groupby("level", as_index=False)["745"]
        .mean()
        .sort_values("level")
    )

    z = profile["level"].values
    T = profile["745"].values

    if len(z) < 20:
        return np.nan, np.nan

    z_uniform = np.arange(z.min(), z.max(), grid_res)
    T_uniform = np.interp(z_uniform, z, T)


    win = min(21, (len(z_uniform)//2)*2-1)

    if win < 5:
        return np.nan, np.nan


    dTdz = savgol_filter(
        T_uniform,
        window_length=win,
        polyorder=2,
        deriv=1,
        delta=grid_res
    )


    # flip sign if needed
    dTdz = -dTdz


    mask = (
        (z_uniform > z_uniform.min()+edge_buffer_m) &
        (z_uniform < z_uniform.max()-edge_buffer_m)
    )


    candidates = np.where(
        mask & (dTdz > threshold)
    )[0]


    if len(candidates) == 0:
        return 0, np.nan


    # strongest point
    idx = candidates[np.argmax(dTdz[candidates])]


    # check vertical persistence
    above = dTdz > threshold

    levels = z_uniform[above]

    close = levels[
        abs(levels-z_uniform[idx]) < min_depth
    ]

    if len(close) < min_depth/grid_res:
        return 0, np.nan

    return dTdz[idx], z_uniform[idx]

strength, height = detect_inversion(df)

if np.isnan(height):
    print("no clear inversion")
else:
    print(strength, height)

soundings_dir = Path("/users/jdelbeke/soundings")

files = sorted(soundings_dir.glob("*_06610_data.txt"))

results=[]

for file in files:

    df = read_sounding(file)

    for termin, sounding in df.groupby("termin"):

        strength, height = detect_inversion(
            sounding,
            threshold=0.01
        )

        results.append({
            "file": file.name,
            "termin": termin,
            "inversion_strength": strength,
            "inversion_height": height
        })


results=pd.DataFrame(results)

results.head()

[20240101000000 20240101120000]
0.04089866117863829 60.0


,file,termin,inversion_strength,inversion_height
0,20240101_06610_data.txt,20240101000000,0.041718,920.0
1,20240101_06610_data.txt,20240101120000,0.048154,60.0
2,20240102_06610_data.txt,20240102000000,0.036509,700.0
3,20240102_06610_data.txt,20240102120000,0.027147,580.0
4,20240103_06610_data.txt,20240103000000,0.036878,60.0


In [51]:
inv = results.dropna(subset=["inversion_strength"]).copy()

# remove non-events if you used 0 for "no inversion"
inv = inv[inv["inversion_strength"] > 0]

inv.head()

p95 = inv["inversion_strength"].quantile(0.95)

print("95th percentile strength:", p95)

strongest_5pct = inv[
    inv["inversion_strength"] >= p95
].sort_values(
    "inversion_strength",
    ascending=False
)


strongest_5pct_date = strongest_5pct.sort_values(
    "termin",
    ascending=True
)

with pd.option_context("display.max_rows", None):
    display(strongest_5pct_date)

95th percentile strength: 0.05619862128146448


,file,termin,inversion_strength,inversion_height
112,20240223_06610_data.txt,20240223120000,0.063701,60.0
172,20240323_06610_data.txt,20240323120000,0.081057,60.0
190,20240401_06610_data.txt,20240401120000,0.057415,60.0
220,20240416_06610_data.txt,20240416000000,0.068983,920.0
221,20240416_06610_data.txt,20240416120000,0.056960,60.0
229,20240420_06610_data.txt,20240420120000,0.058189,60.0
237,20240424_06610_data.txt,20240424120000,0.073922,60.0
241,20240426_06610_data.txt,20240426120000,0.057720,920.0
285,20240517_06610_data.txt,20240517120000,0.070373,60.0
289,20240519_06610_data.txt,20240519120000,0.056199,60.0


In [40]:
sounding = df[df["termin"] == df["termin"].unique()[0]]
profile = sounding[sounding["level"] <= 1000].dropna(subset=["level","745"])
profile = profile.groupby("level", as_index=False)["745"].mean().sort_values("level")
print(len(profile))          # is this ~similar across many files? supports the theory
print(profile.head(15))      # check if T looks reasonable near the surface

999
    level    745
0     0.0  20.00
1     1.0  20.84
2     2.0  21.33
3     3.0  21.49
4     4.0  21.51
5     5.0  21.61
6     6.0  21.80
7     7.0  21.87
8     8.0  21.93
9     9.0  22.11
10   10.0  22.15
11   11.0  22.13
12   12.0  22.13
13   13.0  22.12
14   14.0  22.05
